<a href="https://colab.research.google.com/github/samyuktha07-prog/Interview-AI/blob/main/interviewai.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

import subprocess
import sys
import os
import re
import math
import warnings

warnings.filterwarnings("ignore")

packages = [
    "gradio",
    "transformers",
    "torch",
    "torchaudio",
    "librosa",
    "nltk",
    "scikit-learn",
    "pandas",
    "numpy",
    "matplotlib",
    "soundfile"
]

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q"] + packages,
    check=False
)

import gradio as gr
import numpy as np
import pandas as pd
import librosa
import torch
import nltk

from transformers import (
    pipeline,
    AutoTokenizer,
    AutoModelForSequenceClassification
)

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from collections import Counter

# -----------------------------
# 3. DOWNLOAD NLTK DATA
# -----------------------------
nltk.download("punkt", quiet=True)
nltk.download("vader_lexicon", quiet=True)

from nltk.sentiment import SentimentIntensityAnalyzer

sia = SentimentIntensityAnalyzer()

print("Loading AI models... Please wait.")


try:
    speech_model = pipeline(
        "automatic-speech-recognition",
        model="openai/whisper-tiny.en",
        device=-1
    )
except Exception as e:
    speech_model = None
    print("Speech model loading problem:", e)

try:
    emotion_model = pipeline(
        "text-classification",
        model="j-hartmann/emotion-english-distilroberta-base",
        top_k=None,
        device=-1
    )
except Exception as e:
    emotion_model = None
    print("Emotion model loading problem:", e)

print("Models loaded.")




QUESTIONS = {
    "HR - Tell me about yourself": {
        "question": "Tell me about yourself.",
        "keywords": [
            "education",
            "skills",
            "experience",
            "project",
            "career",
            "goal"
        ]
    },

    "HR - Strengths": {
        "question": "What are your strengths?",
        "keywords": [
            "communication",
            "teamwork",
            "problem solving",
            "leadership",
            "learning",
            "adaptability"
        ]
    },

    "HR - Weakness": {
        "question": "What is your weakness?",
        "keywords": [
            "weakness",
            "improve",
            "learning",
            "challenge",
            "development"
        ]
    },

    "Technical - Python": {
        "question": "Explain your experience with Python.",
        "keywords": [
            "python",
            "programming",
            "function",
            "loop",
            "data",
            "project",
            "library"
        ]
    },

    "Technical - Project": {
        "question": "Explain one of your projects.",
        "keywords": [
            "project",
            "problem",
            "solution",
            "technology",
            "implementation",
            "result"
        ]
    },

    "Technical - Problem Solving": {
        "question": "How do you approach a difficult programming problem?",
        "keywords": [
            "understand",
            "problem",
            "algorithm",
            "break",
            "test",
            "solution",
            "debug"
        ]
    },

    "Behavioral - Teamwork": {
        "question": "Tell me about a time you worked in a team.",
        "keywords": [
            "team",
            "communication",
            "responsibility",
            "collaboration",
            "problem",
            "result"
        ]
    }
}


def clean_text(text):
    if not text:
        return ""

    text = str(text)
    text = text.replace("\n", " ")
    text = re.sub(r"\s+", " ", text)
    return text.strip()



FILLERS = [
    "um",
    "uh",
    "like",
    "you know",
    "actually",
    "basically",
    "literally",
    "so",
    "okay",
    "i mean",
    "kind of",
    "sort of"
]

def analyze_fillers(text):

    lower = text.lower()

    counts = {}

    total = 0

    for filler in FILLERS:
        pattern = r"\b" + re.escape(filler) + r"\b"
        count = len(re.findall(pattern, lower))

        if count > 0:
            counts[filler] = count
            total += count

    return counts, total



def vocabulary_analysis(text):

    words = re.findall(r"\b[a-zA-Z]+\b", text.lower())

    if not words:
        return {
            "total_words": 0,
            "unique_words": 0,
            "vocabulary_ratio": 0,
            "average_word_length": 0
        }

    unique_words = set(words)

    avg_length = sum(len(w) for w in words) / len(words)

    ratio = len(unique_words) / len(words)

    return {
        "total_words": len(words),
        "unique_words": len(unique_words),
        "vocabulary_ratio": ratio,
        "average_word_length": avg_length
    }

def relevance_score(question, answer):

    if not answer.strip():
        return 0

    try:
        vectorizer = TfidfVectorizer(
            stop_words="english"
        )

        vectors = vectorizer.fit_transform([
            question,
            answer
        ])

        similarity = cosine_similarity(
            vectors[0:1],
            vectors[1:2]
        )[0][0]

        return min(100, round(similarity * 180, 2))

    except:
        return 0



def keyword_analysis(answer, keywords):

    answer_lower = answer.lower()

    found = []
    missing = []

    for keyword in keywords:

        if keyword.lower() in answer_lower:
            found.append(keyword)
        else:
            missing.append(keyword)

    if len(keywords) == 0:
        score = 0
    else:
        score = round(
            (len(found) / len(keywords)) * 100,
            2
        )

    return found, missing, score

def sentiment_analysis(text):

    scores = sia.polarity_scores(text)

    compound = scores["compound"]

    if compound >= 0.5:
        label = "Positive"
    elif compound <= -0.3:
        label = "Negative"
    else:
        label = "Neutral"

    return label, scores



def emotion_analysis(text):

    if not text.strip():
        return "Unknown", 0

    if emotion_model is None:
        return "Unavailable", 0

    try:

        results = emotion_model(
            text[:2000]
        )

        if isinstance(results, list) and len(results) > 0:

            data = results[0]

            if isinstance(data, list):

                best = max(
                    data,
                    key=lambda x: x["score"]
                )

            else:
                best = data

            emotion = best["label"]
            confidence = round(
                best["score"] * 100,
                2
            )

            return emotion, confidence

    except Exception as e:
        print("Emotion error:", e)

    return "Neutral", 0



def speech_analysis(audio):

    if audio is None:
        return None, "", 0

    try:

        if isinstance(audio, str):
            audio_path = audio

        else:
            return None, "", 0

        y, sr = librosa.load(
            audio_path,
            sr=None,
            mono=True
        )

        duration = len(y) / sr

        if duration <= 0:
            return None, "", 0

        if speech_model is None:
            transcript = ""
        else:
            result = speech_model(
                audio_path
            )
            transcript = result["text"]

        transcript = clean_text(transcript)

        words = re.findall(
            r"\b[a-zA-Z]+\b",
            transcript
        )

        word_count = len(words)

        minutes = duration / 60

        if minutes > 0:
            wpm = word_count / minutes
        else:
            wpm = 0

        return duration, transcript, round(wpm, 2)

    except Exception as e:

        print("Speech processing error:", e)

        return None, "", 0


def speed_score(wpm):

    if wpm == 0:
        return 0

    if 120 <= wpm <= 160:
        return 100

    if 100 <= wpm < 120:
        return 85

    if 160 < wpm <= 180:
        return 85

    if 80 <= wpm < 100:
        return 70

    if 180 < wpm <= 200:
        return 70

    if wpm < 80:
        return 50

    return 50



def filler_score(filler_count, total_words):

    if total_words == 0:
        return 0

    rate = filler_count / total_words

    if rate <= 0.01:
        return 100

    if rate <= 0.03:
        return 90

    if rate <= 0.06:
        return 75

    if rate <= 0.10:
        return 60

    return 40

def vocabulary_score(vocab_ratio):

    if vocab_ratio >= 0.65:
        return 100

    if vocab_ratio >= 0.55:
        return 90

    if vocab_ratio >= 0.45:
        return 80

    if vocab_ratio >= 0.35:
        return 70

    return 55


def calculate_final_score(
    relevance,
    keyword_score,
    speed,
    filler,
    vocabulary,
    sentiment
):

    sentiment_score = 75

    if sentiment == "Positive":
        sentiment_score = 90
    elif sentiment == "Neutral":
        sentiment_score = 75
    elif sentiment == "Negative":
        sentiment_score = 55

    final = (
        relevance * 0.25 +
        keyword_score * 0.20 +
        speed * 0.15 +
        filler * 0.15 +
        vocabulary * 0.15 +
        sentiment_score * 0.10
    )

    return round(final, 2)


def performance_level(score):

    if score >= 90:
        return "Excellent 🏆"

    if score >= 80:
        return "Very Good 🌟"

    if score >= 70:
        return "Good 👍"

    if score >= 60:
        return "Average 🙂"

    return "Needs Improvement 📚"


def generate_feedback(
    score,
    wpm,
    filler_count,
    relevance,
    keyword_score,
    vocab_score_value
):

    strengths = []
    improvements = []

    if relevance >= 70:
        strengths.append(
            "Your answer is relevant to the interview question."
        )
    else:
        improvements.append(
            "Keep your answer more directly connected to the question."
        )

    if keyword_score >= 70:
        strengths.append(
            "You included several important concepts related to the topic."
        )
    else:
        improvements.append(
            "Include more technical or role-specific keywords."
        )

    if 120 <= wpm <= 160:
        strengths.append(
            "Your speaking speed is suitable for an interview."
        )
    elif wpm < 120 and wpm > 0:
        improvements.append(
            "Try to speak slightly more naturally and confidently."
        )
    elif wpm > 160:
        improvements.append(
            "Try slowing down slightly so the interviewer can follow easily."
        )

    if filler_count <= 2:
        strengths.append(
            "You used very few filler words."
        )
    else:
        improvements.append(
            "Reduce filler words such as 'um', 'like', and 'basically'."
        )

    if vocab_score_value >= 75:
        strengths.append(
            "Your vocabulary variety is good."
        )
    else:
        improvements.append(
            "Try using more precise and varied vocabulary."
        )

    if score >= 80:
        overall = (
            "Strong interview performance. "
            "Focus on maintaining consistency and clarity."
        )
    elif score >= 65:
        overall = (
            "Good foundation. "
            "With more practice, your interview communication can improve significantly."
        )
    else:
        overall = (
            "Keep practicing interview questions. "
            "Focus on relevance, clarity, speaking speed and confidence."
        )

    return strengths, improvements, overall


def analyze_interview(
    question_name,
    audio,
    typed_answer
):

    question_data = QUESTIONS[question_name]

    question = question_data["question"]
    keywords = question_data["keywords"]

    duration = 0
    wpm = 0

    if audio is not None:

        duration, transcript, wpm = speech_analysis(audio)

        if not transcript:
            transcript = clean_text(typed_answer)

    else:
        transcript = clean_text(typed_answer)

    if not transcript:

        return (
            "⚠️ Please record/upload an answer or type your answer.",
            "",
            "",
            "",
            "",
            "",
            None
        )


    words = re.findall(
        r"\b[a-zA-Z]+\b",
        transcript
    )

    total_words = len(words)

    if audio is None:

        if total_words > 0:
            estimated_minutes = total_words / 130
            duration = estimated_minutes * 60
            wpm = 130


    filler_counts, filler_total = analyze_fillers(
        transcript
    )

    filler_rating = filler_score(
        filler_total,
        total_words
    )


    vocab = vocabulary_analysis(
        transcript
    )

    vocab_rating = vocabulary_score(
        vocab["vocabulary_ratio"]
    )



    relevance = relevance_score(
        question,
        transcript
    )


    found_keywords, missing_keywords, keyword_rating = keyword_analysis(
        transcript,
        keywords
    )



    sentiment, sentiment_scores = sentiment_analysis(
        transcript
    )


    emotion, emotion_confidence = emotion_analysis(
        transcript
    )


    speed_rating = speed_score(
        wpm
    )


    final_score = calculate_final_score(
        relevance,
        keyword_rating,
        speed_rating,
        filler_rating,
        vocab_rating,
        sentiment
    )

    level = performance_level(
        final_score
    )


    strengths, improvements, overall = generate_feedback(
        final_score,
        wpm,
        filler_total,
        relevance,
        keyword_rating,
        vocab_rating
    )


    filler_display = ", ".join(
        [f"{k}: {v}" for k, v in filler_counts.items()]
    )

    if not filler_display:
        filler_display = "No major filler words detected"

    found_display = ", ".join(
        found_keywords
    ) if found_keywords else "None"

    missing_display = ", ".join(
        missing_keywords
    ) if missing_keywords else "None"

    strength_display = "\n".join(
        [f"✓ {x}" for x in strengths]
    )

    improvement_display = "\n".join(
        [f"• {x}" for x in improvements]
    )


    report = f"""
# 🎯 InterviewAI Performance Report

## Overall Performance

### {final_score}/100
**Performance Level:** {level}

---

## 📌 Interview Question

**{question}**

---

## 📝 Transcript

> {transcript}

---

## 📊 Detailed Analysis

| Metric | Score |
|---|---:|
| Answer Relevance | {relevance}/100 |
| Keyword Coverage | {keyword_rating}/100 |
| Speaking Speed | {speed_rating}/100 |
| Filler Control | {filler_rating}/100 |
| Vocabulary | {vocab_rating}/100 |

---

## 🎙️ Speech Statistics

**Words:** {total_words}

**Speaking Speed:** {wpm} words/minute

**Duration:** {round(duration, 2)} seconds

**Filler Words:** {filler_total}

**Detected Fillers:** {filler_display}

---

## 🧠 Language Analysis

**Sentiment:** {sentiment}

**Emotion:** {emotion}

**Emotion Confidence:** {emotion_confidence}%

**Unique Words:** {vocab["unique_words"]}

**Vocabulary Diversity:** {round(vocab["vocabulary_ratio"] * 100, 2)}%

---

## 🔑 Interview Keywords

### Detected
{found_display}

### Missing
{missing_display}

---

## 💪 Strengths

{strength_display}

---

## 🚀 Areas to Improve

{improvement_display}

---

## 💡 AI Recommendation

{overall}

---

### InterviewAI
*Text & Speech Analysis based Interview Assessment System*
"""


    score_df = pd.DataFrame({
        "Metric": [
            "Relevance",
            "Keywords",
            "Speaking Speed",
            "Filler Control",
            "Vocabulary"
        ],
        "Score": [
            relevance,
            keyword_rating,
            speed_rating,
            filler_rating,
            vocab_rating
        ]
    })

    return (
        f"## 🎯 Overall Score: {final_score}/100\n\n### {level}",
        report,
        transcript,
        f"{emotion} ({emotion_confidence}%)",
        f"{sentiment}",
        score_df,
        final_score
    )


def sample_answer():

    return (
        "I am a computer science student with a strong interest "
        "in software development and problem solving. I have worked "
        "on academic projects using Python and web technologies. "
        "I enjoy learning new technologies and working with teams. "
        "My goal is to start my career in a product-based technology "
        "company where I can improve my technical skills and contribute "
        "to real-world projects."
    )


css = """
body {
    background: #f5f7fb;
}

.gradio-container {
    max-width: 1200px !important;
    margin: auto !important;
}

.title-box {
    text-align: center;
    padding: 25px;
    border-radius: 18px;
    background: linear-gradient(135deg,#667eea,#764ba2);
    color: white;
    margin-bottom: 20px;
}

.metric-box {
    padding: 20px;
    border-radius: 15px;
    background: white;
    box-shadow: 0 4px 15px rgba(0,0,0,0.08);
}

button {
    border-radius: 10px !important;
}
"""


with gr.Blocks(
    title="InterviewAI",
    css=css,
    theme=gr.themes.Soft()
) as app:

    gr.HTML(
        """
        <div class="title-box">
            <h1>🎯 InterviewAI</h1>
            <h3>Advanced Text & Speech Analysis Interview System</h3>
            <p>
            Analyze your interview answer using AI-powered
            speech recognition, NLP and communication analytics.
            </p>
        </div>
        """
    )

    gr.Markdown(
        """
        ### 🚀 How it works

        **1. Select an interview question →
        2. Record/upload your answer OR type it →
        3. Click Analyze →
        4. Get your AI-powered interview score.**
        """
    )

    with gr.Row():

        with gr.Column(scale=1):

            question_dropdown = gr.Dropdown(
                choices=list(QUESTIONS.keys()),
                value=list(QUESTIONS.keys())[0],
                label="🎤 Select Interview Question"
            )

            question_display = gr.Markdown(
                f"### Question\n{QUESTIONS[list(QUESTIONS.keys())[0]]['question']}"
            )

            audio_input = gr.Audio(
                sources=["microphone", "upload"],
                type="filepath",
                label="🎙️ Record or Upload Your Answer"
            )

            typed_input = gr.Textbox(
                label="📝 Or Type Your Answer",
                placeholder="Type your interview answer here...",
                lines=8
            )

            sample_btn = gr.Button(
                "✨ Load Sample Answer"
            )

            analyze_btn = gr.Button(
                "🚀 ANALYZE INTERVIEW",
                variant="primary"
            )

        with gr.Column(scale=1):

            score_output = gr.Markdown(
                """
                ## 🎯 Your Score

                Submit an answer to see your performance.
                """
            )

            emotion_output = gr.Textbox(
                label="😊 Detected Emotion",
                interactive=False
            )

            sentiment_output = gr.Textbox(
                label="💬 Sentiment",
                interactive=False
            )

            transcript_output = gr.Textbox(
                label="📝 Speech Transcript",
                lines=8,
                interactive=False
            )

    gr.Markdown("---")

    with gr.Row():

        with gr.Column():

            report_output = gr.Markdown(
                "## 📋 Detailed Report\nYour report will appear here."
            )

        with gr.Column():

            score_table = gr.Dataframe(
                headers=["Metric", "Score"],
                label="📊 Performance Metrics",
                interactive=False
            )


    def update_question(choice):

        return (
            f"### Question\n{QUESTIONS[choice]['question']}"
        )

    question_dropdown.change(
        update_question,
        inputs=question_dropdown,
        outputs=question_display
    )


    sample_btn.click(
        sample_answer,
        outputs=typed_input
    )


    analyze_btn.click(
        analyze_interview,
        inputs=[
            question_dropdown,
            audio_input,
            typed_input
        ],
        outputs=[
            score_output,
            report_output,
            transcript_output,
            emotion_output,
            sentiment_output,
            score_table,
            gr.State()
        ]
    )


print("\n" + "=" * 60)
print("INTERVIEWAI IS READY")
print("=" * 60)
print("Use the microphone or upload an audio file.")
print("You can also type an answer.")
print("=" * 60)

app.launch(
    share=True,
    debug=True
)

Loading AI models... Please wait.


config.json:   0%|          | 0.00/1.94k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  151MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/167 [00:00<?, ?it/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  329MB            

model.safetensors: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Models loaded.

INTERVIEWAI IS READY
Use the microphone or upload an audio file.
You can also type an answer.
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://361eadd098826bc2c8.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
